# 4. 다중 경로 workflow 평가

**시나리오:** billing·access·fallback 티켓을 같은 평가표로 비교합니다.

**학습 목표:** 단일 happy path가 아니라 category·route·citation의 일관성을 회귀 테스트하는 방법을 익힙니다.

## 중요 변수·함수

- `cases`: 입력과 기대 route를 한 쌍으로 관리하는 작은 평가 fixture입니다.
- `citations`: 분류 category와 같은 playbook인지 확인합니다.
- `status`: 근거가 있으면 `plan_ready`, 없으면 `needs_more_information`으로 구분합니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 세 업무 경로를 canonical workflow로 반복 실행합니다.
from week2.app import TicketRequest, create_fixture_services, run_ticket_workflow

cases = [
    (TicketRequest(subject='Duplicate invoice', description='Charged twice', customer_tier='standard'), 'billing_queue', 'plan_ready'),
    (TicketRequest(subject='Login problem', description='Account login fails', customer_tier='standard'), 'access_queue', 'plan_ready'),
    (TicketRequest(subject='General question', description='Please explain pricing', customer_tier='standard'), 'general_queue', 'needs_more_information'),
]
results = [(run_ticket_workflow(ticket, create_fixture_services()), route, status) for ticket, route, status in cases]

In [ ]:
# route와 read-only 상태를 평가합니다.
for result, expected_route, expected_status in results:
    assert result['route'] == expected_route
    assert result['status'] == expected_status
    assert result['mode'] == 'fixture'
[(result['category'], result['route'], result['citations']) for result, _, _ in results]

## 예측 과제와 해석

**예측 과제:** fallback 티켓에 citation이 없을 때 그것이 반드시 실패인지 판단 기준을 적으세요.

**해석:** fallback은 citation 없이 모델 계획을 만들지 않고 `needs_more_information`으로 종료합니다. 시스템은 환불·계정 변경·티켓 종료를 실행했다고 주장하지 않습니다.